# recheck_comlumn_preapare — ตรวจ layer + แตก process parameter

recheck ผลของ `column_prepare.ipynb` อ่านจาก `processed/prepared/` (ไม่แก้ต้นทาง)

4 เรื่อง
1. layer หลัก vs layer เคลือบผิว + จำนวนชั้น
2. คอลัมน์ใหม่ระดับ device: เคลือบ / ไม่เคลือบ (layer 3,5,7 = ชั้นเคลือบ)
3. layer 2,4 ปิดด้วย Annealing จริงไหม
4. แตก `process.parameter` + `unit` (เช่น `155, 600, 600` / `W, mtorr, sec`) เป็นคอลัมน์แยก

## Setup

In [ ]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 100)

ROOT = Path.cwd() if (Path.cwd() / "processed" / "prepared").exists() else Path.cwd().parent
SRC = ROOT / "processed" / "prepared"
tp = {p.stem: pd.read_csv(p) for p in sorted(SRC.glob("*.csv"))}
lay, pr = tp["layers"], tp["process"]
print("โหลดจาก:", SRC)
display(pd.DataFrame({n: d.shape for n, d in tp.items()}, index=["rows", "cols"]).T)

## 1. layer หลัก vs layer เคลือบผิว

ตำแหน่ง layer_id เป็นระบบ: **เลขคู่ = ชั้นหลัก, เลขคี่กลาง = ชั้นเคลือบ** (แทรกระหว่างชั้นหลัก)

**ชั้นเคลือบ N เคลือบ "ผิวบนของชั้นหลักก่อนหน้า (N−1)" เสมอ** — ตามลำดับการ deposit จริง (สร้างจากล่างขึ้นบน) และไปทำงานที่**รอยต่อ**ระหว่างชั้น N−1 กับ N+1 ยืนยันจากวัสดุจริงใน dataset:

| layer_id | บทบาท | ชนิด | เคลือบผิวของ | วัสดุจริงที่เจอ + หน้าที่ |
|---|---|---|---|---|
| 1 | Anode | หลัก | — | ITO/FTO |
| 2 | HTL | หลัก | — | NiOx |
| **3** | **HTL treatment** | **เคลือบ** | **L2 (HTL)** | SAM (Me-2PACz/4PACz), Al2O3, ionic liquid — ปรับผิว NiOx ก่อนวาง PSK (buried interface: ช่วย wetting/การโตผลึกของชั้นบน) |
| 4 | PSK | หลัก | — | perovskite ABX3 |
| **5** | **PSK treatment** | **เคลือบ** | **L4 (PSK)** | PEAI 15/22 dev — passivation ผิวบน perovskite เกิดชั้น 2D อุด defect ก่อนวาง ETL |
| 6 | ETL | หลัก | — | PCBM/C60 |
| **7** | **ETL treatment** | **เคลือบ** | **L6 (ETL)** | BCP 37/44 dev หนา ~9 nm — buffer กัน metal ทะลุ + เปลี่ยน Schottky เป็น ohmic contact ที่รอยต่อกับขั้ว |
| 8 | Cathode | หลัก | — | Ag/Au/Cu |

หมายเหตุ: จะพูดว่า "เคลือบชั้นล่าง" (ตาม deposit) หรือ "interface layer ของรอยต่อบน" (ตามหน้าที่) ก็ถูกทั้งคู่ — คนละมุมของสิ่งเดียวกัน

In [ ]:
MAIN_IDS    = {1, 2, 4, 6, 8}
COATING_IDS = {3, 5, 7}

_L = lay.dropna(subset=["material"]).copy()
_L["layer_kind"] = _L["layer_id"].map(
    lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")

print("layer_id x layer_type (นับ device) + ชนิดที่จัดให้:")
_chk = (lay.groupby(["layer_id", "layer_type"]).device_id.nunique().rename("n_dev").reset_index())
_chk["kind"] = _chk["layer_id"].map(lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")
display(_chk)

print("จำนวนชั้น (นับเฉพาะ material ไม่ว่าง) ต่อ device:")
_cnt = _L.groupby("device_id").agg(
    n_total=("layer_id", "nunique"),
    n_main=("layer_kind", lambda s: (s == "main").sum()),
    n_coating=("layer_kind", lambda s: (s == "coating").sum()))
display(_cnt["n_coating"].value_counts().sort_index().rename("n_devices").to_frame())

## 2. คอลัมน์ใหม่: เคลือบ / ไม่เคลือบ (ระดับ device)

ธงว่าแต่ละ device มีชั้นเคลือบตรงตำแหน่งไหนบ้าง + สรุปรวม

**§2.1 แยก "เคลือบผิวจริง" vs "additive ผสมในน้ำหมึก"** — ช่อง L3/L5 ใน `layers` ถูกใช้บันทึก 2 อย่างที่ต่างกันทางกายภาพ:
- **coating** — เคลือบทับผิวชั้นล่าง มีขั้นตอน deposit ของตัวเอง (`process` ที่ `layer_id` เดียวกัน)
- **additive** — สารถูกผสมลงในน้ำหมึกของชั้นหลักแล้ว spin ครั้งเดียว ไม่มี process ที่ L3/L5 แต่ไปโผล่เป็นขั้น `Mix` ในชั้นหลักแทน

เกณฑ์แยกแบบ data-driven: มี material ที่ L3/L5 **แต่ไม่มี process** = additive (ชั้นเคลือบจริงต้องมีขั้นตอน deposit เสมอ)

In [ ]:
dev = tp["devices"][["device_id"]].copy()

# ---- 2.1 ชนิดการดัดแปลงที่ L3/L5: เคลือบทับ vs ผสมในน้ำหมึก ----
# ชั้นเคลือบจริงต้องมีขั้นตอน deposit ของตัวเอง (process ที่ layer_id เดียวกัน)
# มี material ที่ L3/L5 แต่ไม่มี process เลย = สารถูกผสมลงชั้นหลักแทน ไม่ได้เคลือบทับ
_Pn = pr[pr["process_type"].notna()]        # กัน placeholder แถวเปล่า (28 แถว) นับเกิน
MOD_KIND = {3: "htl_mod_kind", 5: "psk_mod_kind"}
for lid, col in MOD_KIND.items():
    has_mat  = dev["device_id"].isin(_L.loc[_L["layer_id"] == lid, "device_id"])
    has_proc = dev["device_id"].isin(_Pn.loc[_Pn["layer_id"] == lid, "device_id"])
    dev[col] = np.where(~has_mat, None, np.where(has_proc, "coating", "additive"))

print("ชนิดการดัดแปลง (additive = ผสมในน้ำหมึกชั้นหลัก ไม่ใช่เคลือบทับ):")
display(pd.DataFrame({c: dev[c].value_counts(dropna=False) for c in MOD_KIND.values()}))
_add = dev[dev[list(MOD_KIND.values())].eq("additive").any(axis=1)]["device_id"]
if len(_add):
    print("device ที่เป็น additive:", list(_add))
    print("หลักฐาน -- ขั้นตอน Mix ในชั้นหลักที่ระบุชื่อสาร:")
    display(_Pn[_Pn["device_id"].isin(_add) & _Pn["process_type"].str.contains("Mix", case=False, na=False)]
            [["device_id", "layer_id", "step_order", "process_type", "description"]])

# ---- ธงเคลือบ: นับเฉพาะ "เคลือบทับจริง" (additive ไม่นับ -- ไม่ได้เพิ่มชั้น) ----
COAT_MAP = {3: "coat_htl", 5: "coat_psk", 7: "coat_etl"}
for lid, col in COAT_MAP.items():
    if lid in MOD_KIND:                      # L3/L5 ใช้ผลจาก mod_kind
        dev[col] = dev[MOD_KIND[lid]].eq("coating")
    else:                                    # L7 ไม่มีเคส additive
        dev[col] = dev["device_id"].isin(_L.loc[_L["layer_id"] == lid, "device_id"])

dev["is_coated"] = dev[list(COAT_MAP.values())].any(axis=1)
dev["n_coating_layers"] = dev[list(COAT_MAP.values())].sum(axis=1)
assert dev["coat_htl"].eq(dev["htl_mod_kind"].eq("coating")).all(), "coat_htl ต้องตรงกับ htl_mod_kind"

print()
print("device เคลือบ / ไม่เคลือบ:")
display(dev["is_coated"].value_counts().rename("n_devices").to_frame())
print("แยกตามตำแหน่ง (coat_htl/psk = เคลือบทับจริง ไม่รวม additive):")
display(dev[list(COAT_MAP.values())].sum().rename("n_devices").to_frame())
display(dev.head())

## 3. layer 2, 4 ปิดด้วย Annealing จริงไหม

เช็คจาก `process` (`source_sheet == "Process"`) ว่า layer 2 (HTL) และ 4 (PSK) มีขั้น Annealing

In [ ]:
_P = pr[pr["source_sheet"] == "Process"]

for lid, name in [(2, "HTL"), (4, "PSK")]:
    g = _P[_P["layer_id"] == lid]
    ann = g[g["process_type"].str.contains("nneal", case=False, na=False)]
    print(f"layer {lid} ({name}): {ann.device_id.nunique()}/{g.device_id.nunique()} devices มี Annealing")

# ธงระดับ device
for lid, col in [(2, "anneal_htl"), (4, "anneal_psk")]:
    hit = _P[(_P["layer_id"] == lid) & _P["process_type"].str.contains("nneal", case=False, na=False)]["device_id"].unique()
    dev[col] = dev["device_id"].isin(hit)

display(dev[["anneal_htl", "anneal_psk"]].sum().rename("n_devices").to_frame())

## 4. แตก `process.parameter` + `unit` เป็นคอลัมน์แยก

`parameter` กับ `unit` เป็นค่าหลายตัวคั่น comma จับคู่ตามตำแหน่ง

```
parameter = "155, 600, 600"   unit = "W, mtorr, sec"
    -> W=155   mtorr=600   sec=600
```

ชื่อคอลัมน์ = `process_type` + `unit` เพราะหน่วยเดียวกันคนละความหมายตามขั้นตอน
(`C` ใน Annealing กับใน Dry คนละเรื่อง)

**⚠️ 9 แถว parameter พิมพ์ผิด** (ตัวเลขติดกัน / comma หาย) — 8 แถวแก้ใน §4.0 ด้วยค่ายืนยันจาก sibling sample ใน Raw Excel, เหลือ 1 แถวกำกวมติดธง `param_note` ไว้ ไม่เดา

### 4.0 แก้ parameter พิมพ์ผิดก่อนแตก

| ดิบ | แก้เป็น | หลักฐาน |
|---|---|---|
| `155600600` | `155, 600, 600` | S3 ไฟล์เดียวกันเขียน `155,600, 600` |
| `1556003600` | `155, 600, 3600` | รูปแบบเดียวกัน plasma นานขึ้น |
| `27010800` | `270, 10800` | S1 เขียน `270, 10800` |
| `603600` | `60, 3600` | S1 เขียน `60, 3600` |
| `80, 0.03 500` | `80, 0.03, 500` | comma หายหลัง 0.03 |

**ไม่แก้**: `100108000` (admi S3, L4 Annealing) — S1=`100, 1800` S4=`100, 10800` ขัดกัน รอเปิด paper ไม่เดา

In [ ]:
# ---- §4.0 แก้ parameter พิมพ์ผิด (ยืนยันจาก sibling sample ใน Raw Excel ไฟล์เดียวกัน) ----
# ค่าทั้งหมดของ 10.1002admi.202500231 ยืนยันจาก Supporting Information (admi70014-sup-0001) §1.2
#   "sonicated in acetone followed by isopropanol for 10 minutes each"       -> Wash 1200 s
#   "O2-plasma treatment at 155 W with 600 mtorr O2 ... for 10 min"          -> 155, 600, 600
#   "spin-coating at 3000 rpm for 30 s" (NiOx)                               -> 3000, 30
#   "mixed on a hot plate at 60 C ... during 1 h"                            -> 60, 3600
#   "1000 rpm ... 500 rpm/s ... 10 s / 4000 rpm ... 2000 rpm/s ... 35 s"     -> ตรงอยู่แล้ว
#   "At 10 s before the end ... 0.25 ml of chlorobenzene was dropped"        -> 0.25, -10 (ยืนยัน convention ค่าลบ)
#   "put on a hot plate at 100 C for 30 min"                                 -> 100, 1800  <- ทั้ง 4 sample
PARAM_FIX = {
    # (device_id, layer_id, ค่าดิบ): ค่าที่ถูก
    ("10.1002admi.202500231_Sample1", 1, "155600600"):     "155, 600, 600",   # SI: 155 W, 600 mtorr, 10 min
    ("10.1002admi.202500231_Sample4", 1, "1556003600"):    "155, 600, 600",   # SI: 10 min ทุก sample (ไม่ใช่ 3600 s)
    ("10.1002admi.202500231_Sample3", 2, "27010800"):      "270, 10800",      # ดู NOTE_L2_ANNEAL ด้านล่าง
    ("10.1002admi.202500231_Sample4", 2, "27010800"):      "270, 10800",
    ("10.1002admi.202500231_Sample3", 4, "603600"):        "60, 3600",        # SI: 60 C, 1 h
    ("10.1002admi.202500231_Sample4", 4, "603600"):        "60, 3600",
    # --- PSK annealing: SI ระบุ "100 C for 30 min" = 1800 s เท่ากันทั้ง 4 sample ---
    #     (สูตร perovskite เหมือนกันหมด ต่างแค่ EMIMI ที่ NiOx -- SI ยืนยัน "same batch")
    ("10.1002admi.202500231_Sample2", 4, "100, 108000"):   "100, 1800",
    ("10.1002admi.202500231_Sample3", 4, "100108000"):     "100, 1800",       # เคสที่เคยติดธง -- ปิดแล้ว
    ("10.1002admi.202500231_Sample4", 4, "100, 10800"):    "100, 1800",
    ("10_1021_acsami_4c14300_Sample1", 2, "80, 0.03 500"): "80, 0.03, 500",   # comma หายหลัง 0.03
    ("10_1021_acsami_4c14300_Sample2", 2, "80, 0.03 500"): "80, 0.03, 500",
}
# NOTE_L2_ANNEAL: SI §1.2 (device fabrication) ไม่มีขั้นอบฟิล์ม NiOx เลย -- 270 C 3 h เป็นการ "เผาผง"
#   ใน §1.1 Materials synthesis ("The as-prepared green solid was then annealed at 270 C for 3 h")
#   แถว Annealing ที่ L2 ใน Process sheet จึงซ้ำกับ NiOx synthesis sheet -> htl_anneal_C ของ 4 device นี้
#   ไม่ใช่ขั้นตอนสร้างเซลล์จริง (ดู §9.1 ที่กรองออก)

pr["param_fixed"] = False
_lidn = pd.to_numeric(pr["layer_id"], errors="coerce")
for (d, l, bad), good in PARAM_FIX.items():
    m = (pr["device_id"] == d) & (_lidn == l) & (pr["parameter"].astype(str).str.strip() == bad)
    assert m.sum() == 1, f"PARAM_FIX จับได้ {int(m.sum())} แถว (ต้อง 1): {d} L{l} {bad!r}"
    pr.loc[m, "parameter"] = good
    pr.loc[m, "param_fixed"] = True

print(f"แก้แล้ว {int(pr['param_fixed'].sum())}/11 แถว -- ทุกค่ายืนยันจาก Supporting Information แล้ว ไม่เหลือแถวกำกวม")

In [ ]:
UNIT_FIX = {"µL": "uL", "min each": "min"}          # รวมหน่วยพ้อง

def split_param(param, unit):
    """คืน dict {unit: value} + note ถ้าจำนวนไม่ตรง"""
    if pd.isna(param):
        return {}, ""
    ps = [t.strip() for t in str(param).split(",") if t.strip()]
    us = [UNIT_FIX.get(t.strip(), t.strip()) for t in str(unit).split(",") if t.strip()] if pd.notna(unit) else []
    if not us:
        return {}, ""
    if len(ps) != len(us):
        return {}, f"param {len(ps)} ค่า != unit {len(us)} ค่า: {param!r} | {unit!r}"
    out = {}
    for p, u in zip(ps, us):
        out[u] = pd.to_numeric(p, errors="coerce")
    return out, ""

parsed, notes = [], []
for _, r in pr.iterrows():
    d, note = split_param(r["parameter"], r["unit"])
    parsed.append(d)
    notes.append(note)

_pv = pd.json_normalize(parsed)                                  # แต่ละ unit เป็นคอลัมน์
_pv.columns = [f"{pt}__{u}" for pt, u in
               zip(pr["process_type"].fillna("NA"), [""] * len(_pv))] if False else _pv.columns
# ตั้งชื่อ process_type + unit ต่อแถว: ทำผ่านการ melt เพื่อความชัด
long = []
for i, (d, note) in enumerate(zip(parsed, notes)):
    ptype = pr.iloc[i]["process_type"]
    for u, v in d.items():
        long.append({"row": i, "col": f"{ptype}__{u}", "value": v})
proc_split = pd.DataFrame(long)

pr_out = pr.copy()
pr_out["param_note"] = notes
wide = proc_split.pivot_table(index="row", columns="col", values="value", aggfunc="first")
pr_out = pr_out.join(wide)

n_bad = int((pr_out["param_note"] != "").sum())
n_cols = wide.shape[1]
print(f"แตกเป็น {n_cols} คอลัมน์ | {n_bad} แถวติดธง (param/unit ไม่ตรง)")

print("\nแถวที่ติดธง (ตัวเลขติดกัน ต้องแก้ที่ต้นฉบับ):")
display(pr_out.loc[pr_out["param_note"] != "", ["device_id", "process_type", "parameter", "unit", "param_note"]]
        .drop_duplicates(subset=["parameter", "unit"]))

### 4.1 ตัวอย่างคอลัมน์ที่แตกได้ — process knob ที่เอาไปใช้ต่อได้

In [ ]:
_knob_cols = [c for c in pr_out.columns if "__" in c]
_ns = pr_out["source_sheet"] == "NiOx synthesis"     # สูตรทำผง NiOx ไม่ใช่ขั้นตอนสร้างเซลล์
print("คอลัมน์ทั้งหมด  (!! = มาจาก NiOx synthesis ล้วน -- confound กับ DOI เดียว ห้ามเป็น feature)")
for c in sorted(_knob_cols):
    n = int(pr_out[c].notna().sum())
    if n:
        n_ns = int((pr_out[c].notna() & _ns).sum())
        tag = " !!" if n_ns == n else (f"  ({n_ns} แถวจาก NiOx synthesis)" if n_ns else "")
        print(f"  {c:28s} {n:3d} แถว  ({pr_out[c].min():g} - {pr_out[c].max():g}){tag}")

print("\nตัวอย่าง Spin coating + Annealing:")
_show = [c for c in _knob_cols if c.startswith(("Spin coating", "Annealing"))]
display(pr_out.loc[pr_out[_show].notna().any(axis=1),
                   ["device_id", "layer_id", "process_type", "parameter", "unit"] + _show].head(10))

## 5. สรุป device-level ที่ได้เพิ่ม

รวมธง layer เข้า `dev` — เอาไป join กับ `ml` ใน `after_col_prepare` ได้

In [ ]:
cols = ["device_id", "is_coated", "n_coating_layers",
        "coat_htl", "coat_psk", "coat_etl", "anneal_htl", "anneal_psk",
        "htl_mod_kind", "psk_mod_kind"]        # §2.1 coating vs additive
device_layer_flags = dev[cols]
print("device_layer_flags:", device_layer_flags.shape)
display(device_layer_flags.head())
# ยังไม่เขียน -- รวมเขียนทีเดียวใน §10

## 6. normalize เวลาเป็นวินาที + หมายเหตุ anneal temp

หน่วยเวลาปนกัน `sec` / `min` / `hr` รวมเป็น `sec` เดียว (`hr x3600`, `min x60`)

**anneal temp สูง (270-450C) ไม่ใช่บั๊ก** — เป็นการ anneal ชั้น NiOx/HTL (ปกติ 270-450C)
คนละตัวกับ anneal ชั้น PSK (100-150C) **ต้องแยกด้วย `layer_id` ตอนใช้เป็น feature** ห้ามรวม

In [ ]:
SEC_PER = {"sec": 1, "min": 60, "hr": 3600}

# รวมเวลาต่อ process_type: <ptype>__time_s = sec + min*60 + hr*3600
_time_units = ["sec", "min", "hr"]
_ptypes = sorted({c.split("__")[0] for c in pr_out.columns if "__" in c})
for pt in _ptypes:
    parts = [(f"{pt}__{u}", SEC_PER[u]) for u in _time_units if f"{pt}__{u}" in pr_out.columns]
    if len(parts) > 1 or (parts and parts[0][0].split("__")[1] != "sec"):
        s = pd.Series(0.0, index=pr_out.index)
        any_val = pd.Series(False, index=pr_out.index)
        for col, mult in parts:
            s = s + pr_out[col].fillna(0) * mult
            any_val = any_val | pr_out[col].notna()
        pr_out[f"{pt}__time_s"] = s.where(any_val)

# ตรวจ anneal แยกตาม layer
_ann = pr_out[pr_out["process_type"] == "Annealing"].dropna(subset=["Annealing__C"])
print("anneal temp แยกตาม layer_id (เห็นว่าชั้นล่าง NiOx/HTL ร้อนกว่า PSK):")
display(_ann.groupby("layer_id")["Annealing__C"].agg(["count", "min", "median", "max"]))

### 6.1 AntiSolvent: รวมหน่วยปริมาตร + convention เวลา

- `__ml` กับ `__uL` เป็น knob เดียวกัน → `AntiSolvent__vol_uL` (0.25 ml = 250 uL)
- เวลาหยด: **ค่าลบ = วินาทีก่อนจบ spin, ค่าบวก = วินาทีหลังเริ่ม** คนละจุดอ้างอิง
  เก็บ `AntiSolvent__timing_ref` กำกับ ค่าดิบห้ามรวมสเกลตรง ๆ
  → §9 แปลงรวมเป็นแกนเดียวแล้ว: `antisolvent_drop_s` = วินาทีนับจากเริ่ม spin ขั้นสุดท้าย
  (ตัวลบแปลงด้วย `psk_spin_last_s + timing`) + `antisolvent_drop_frac` = สัดส่วนของช่วง spin

In [ ]:
# ---- §6.1 AntiSolvent: รวมหน่วยปริมาตร + convention เวลา ----
# ปริมาตร: __ml (4 dev) กับ __uL (31 dev) เป็น knob เดียวกัน -> uL เดียว
vol = pd.Series(np.nan, index=pr_out.index)
if "AntiSolvent__uL" in pr_out.columns:
    vol = pr_out["AntiSolvent__uL"].copy()
if "AntiSolvent__ml" in pr_out.columns:
    vol = vol.fillna(pr_out["AntiSolvent__ml"] * 1000)
pr_out["AntiSolvent__vol_uL"] = vol

# เวลา: ค่าลบ = วินาที "ก่อนจบ" spin, ค่าบวก = วินาที "หลังเริ่ม" -- คนละจุดอ้างอิง ห้ามรวมสเกลตรง ๆ
t = pr_out["AntiSolvent__sec"] if "AntiSolvent__sec" in pr_out.columns else pd.Series(np.nan, index=pr_out.index)
pr_out["AntiSolvent__timing_s"] = t
pr_out["AntiSolvent__timing_ref"] = pd.Series(
    np.where(t < 0, "before_end", "after_start"), index=pr_out.index).where(t.notna())

_m = pr_out["AntiSolvent__vol_uL"].notna()
print(f"AntiSolvent__vol_uL: {pr_out.loc[_m, 'device_id'].nunique()} devices "
      f"(uL {int(pr_out['AntiSolvent__uL'].notna().sum())} แถว + ml {int(pr_out['AntiSolvent__ml'].notna().sum())} แถว)")
print("timing_ref:", pr_out["AntiSolvent__timing_ref"].value_counts().to_dict())

## 7. ฝั่ง Y: เลือก target จาก `properties`

**2 ชั้น แยกกัน**

**Y ประสิทธิภาพ (target หลัก)** — เทรน Voc/Jsc/FF แยกแล้วคูณเป็น PCE
เพราะ `PCE = Voc x Jsc x FF / 100` (พิสูจน์แล้ว corr 0.996) เทรน PCE ตรงไม่ได้ข้อมูลใหม่ แยกได้กลไก

**mediator เชิงกายภาพ** — ใช้ **VTFL** เป็นตัวหลัก (29 dev เยอะกว่า grain_size 13 และ trap_density 23)
VTFL วัดข้อบกพร่องเหมือน trap density แต่เป็นค่าดิบ coverage ดีกว่า

In [ ]:
prop = tp["properties"].dropna(subset=["value_parsed"]).copy()
prop["paper"] = prop["device_id"].str.rsplit("_Sample", n=1).str[0]

TARGETS  = ["PCE", "Voc", "Jsc", "FF"]
MEDIATOR = ["VTFL", "Trap density", "Rs", "Rrec", "Contact angle", "Grain size"]

cov = (prop[prop["base_property"].isin(TARGETS + MEDIATOR)]
       .groupby("base_property")
       .agg(rows=("value_parsed", "size"),
            devices=("device_id", "nunique"),
            papers=("paper", "nunique"),
            lo=("value_parsed", "min"), hi=("value_parsed", "max")))
cov["บทบาท"] = cov.index.map(lambda b: "target" if b in TARGETS else "mediator")
cov["ใช้"] = cov["devices"].ge(15) & cov["papers"].ge(5)
display(cov.sort_values(["บทบาท", "devices"], ascending=[True, False]))

# ---- JV: เลือกชุดเดียวกัน (variant-consistent) ----
# บั๊กเดิม: aggfunc="max" ต่อ property อิสระ -> บาง paper ได้ Champion PCE ปน Average Voc/Jsc/FF
# (เช่น d3lf00258f รายงาน component เป็น Average อย่างเดียว) ทำให้ pce != voc*jsc*ff/100
_jv = prop[prop["base_property"].isin(TARGETS)].copy()
_p = _jv["property"].fillna("")
_jv["variant"] = np.where(_p.str.contains("hampion"), "champion",
                 np.where(_p.str.contains("verage"), "average", "plain"))

piv = {v: (_jv[_jv["variant"] == v]
           .pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="max")
           .reindex(columns=TARGETS))
       for v in ["champion", "plain", "average"]}

rows = []
for dev in sorted(_jv["device_id"].unique()):
    pick, r = None, None
    for v in ["champion", "plain", "average"]:          # ชุดแรกที่มี voc+jsc+ff ครบ
        if dev in piv[v].index and piv[v].loc[dev, ["Voc", "Jsc", "FF"]].notna().all():
            pick, r = v, piv[v].loc[dev]
            break
    if pick is None:                                     # ไม่มีชุดครบใน variant เดียว -> รวมทุกแถว ติดธง
        pick = "mixed"
        r = (_jv[_jv["device_id"] == dev]
             .pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="max")
             .reindex(columns=TARGETS).loc[dev])
    rows.append({"device_id": dev, "jv_variant": pick,
                 **{c.lower(): r[c] for c in TARGETS}})
jv = pd.DataFrame(rows).set_index("device_id")

print("jv_variant:", jv["jv_variant"].value_counts().to_dict())
_have = jv.dropna(subset=["voc", "jsc", "ff", "pce"])
_err = (_have["voc"] * _have["jsc"] * _have["ff"] / 100 - _have["pce"]).abs()
# เกณฑ์ 1.0: ค่ารายงานปัด 2 ตำแหน่ง ทำ product คลาดได้ ~0.5 (เช่น ccschem champion) -- ไม่ใช่ปน variant
print(f"เช็ค pce = voc*jsc*ff/100: n={len(_have)} max_err={_err.max():.3f} | เกิน 1.0: {int((_err > 1.0).sum())} เครื่อง (ควรเป็น 0)")

# mediator ที่ผ่านเกณฑ์
med = (prop[prop["base_property"].isin(["VTFL", "Trap density", "Rs", "Rrec"])]
       .pivot_table(index="device_id", columns="base_property", values="value_parsed", aggfunc="median"))
med.columns = [c.lower().replace(" ", "_") for c in med.columns]

Y = jv.join(med)
print(f"\nตาราง Y: {Y.shape[0]} devices x {Y.shape[1]} คอลัมน์")
display(Y.describe(include=[np.number]).T.round(3))

## 8. Stability: คิด index ก่อนใช้เป็น Y

retention ดิบใช้ตรงไม่ได้ — วัดคนละสภาพ (25-85C, 1-5000 ชม.)

ทำ 2 อย่าง
1. `degradation_rate` = `(100 - retention) / time_hr` [%/ชม.] — normalize เวลา
2. `temp_bin` — จับกลุ่มอุณหภูมิ ใช้เทียบเฉพาะในกลุ่มเดียวกัน

**ข้อจำกัด**: การเสื่อมไม่เชิงเส้น (burn-in เร็วแล้วช้า) rate เทียบข้าม timescale ตรง ๆ ไม่ได้
ใช้ retention @ 25C bin เป็น Y เสถียร (14 dev) เป็นทางที่สะอาดสุด แต่ยัง exploratory

In [ ]:
stab = tp["stability"].copy()
stab["ret"] = pd.to_numeric(stab["retention_pct_parsed"], errors="coerce")
stab["t_hr"] = pd.to_numeric(stab["time_hr"], errors="coerce")
stab["Tp"] = pd.to_numeric(stab["temperature_c_parsed"], errors="coerce")
stab = stab.dropna(subset=["ret"])

stab["degradation_rate"] = (100 - stab["ret"]) / stab["t_hr"]         # %/hr
stab["temp_bin"] = pd.cut(stab["Tp"], [0, 30, 60, 100],
                          labels=["~25C", "45-50C", "65-85C"])

print("index รายกลุ่มอุณหภูมิ:")
display(stab.groupby("temp_bin", observed=True).agg(
    devices=("device_id", "nunique"),
    ret_median=("ret", "median"),
    rate_median=("degradation_rate", "median")).round(3))
print(f"ไม่มี temp: {int(stab['Tp'].isna().sum())} แถว (ใช้ได้เฉพาะ degradation_rate)")

# Y เสถียร: retention ที่ ~25C bin (สภาพใกล้กันสุด)
stab_Y = (stab[stab["temp_bin"] == "~25C"]
          .groupby("device_id").agg(retention_25c=("ret", "max"),
                                    degradation_rate=("degradation_rate", "min")))
print(f"\nY เสถียร (retention @ 25C): {len(stab_Y)} devices")
display(stab_Y.head())

## 8.5 PSK composition (a_Cs / a_FA / x_Br)

`column_prepare` §J คำนวณครบ 44 devices แต่**ไม่เคย export ลงดิสก์** — port ตรรกะเดิมทุกบรรทัด (J1-J5, TOL=0.03) มาคำนวณใหม่จากตาราง prepared ที่นี่

สูตรนามเป็นหลัก → precursor เติมช่องว่าง + cross-check | export เป็น `psk_composition.csv` ใน §10

In [ ]:
# ---- §8.5 psk_composition -- port ตรรกะเดิมจาก column_prepare §J (J1-J5) ----
# column_prepare คำนวณแล้วแต่ไม่เคยเขียนดิสก์ -- คำนวณใหม่จากตาราง prepared ที่นี่แล้ว export ใน §10

# J1: อ่านสัดส่วนจากสูตรในชื่อชั้น
CATIONS = ["FA", "MA", "Cs", "Rb", "Pb", "Sn", "Br", "Cl", "I", "K"]   # ยาว->สั้น กัน FA ถูกอ่านเป็น F+A
TOKEN = re.compile("(" + "|".join(CATIONS) + r")(\d*\.?\d+)")

def _site_from_str(part):
    out = {}
    for inner, scale in re.findall(r"\(([^)]*)\)\s*(\d*\.?\d+)?", part):
        k = float(scale) if scale else 1.0
        for ion, frac in TOKEN.findall(inner):
            out[ion] = out.get(ion, 0) + float(frac) * k
    rest = re.sub(r"\([^)]*\)\s*\d*\.?\d*", "", part)
    for ion, frac in TOKEN.findall(rest):
        out[ion] = out.get(ion, 0) + float(frac)
    return out

def composition_from_formula(f):
    s = str(f).strip().strip("[]").strip()
    m = re.search(r"(Pb|Sn)", s)
    if not m:
        return None
    A, X = _site_from_str(s[:m.start()]), _site_from_str(s[m.end():])
    sA, sX = sum(A.values()), sum(X.values())
    if not sA or not sX:
        return None
    return ({k: v / sA for k, v in A.items()}, {k: v / sX for k, v in X.items()}, m.group(1))

# J2: molar mass + ไอออนที่สารตั้งต้นให้
MW = {"CsI": 259.81, "CsBr": 212.81, "FAI": 171.97, "MAI": 158.97, "MABr": 111.97,
      "MACl": 67.52, "PbI2": 461.01, "PbBr2": 367.01, "PbCl2": 278.11}
ION = {"CsI":  {"A": {"Cs": 1}, "X": {"I": 1}},   "CsBr": {"A": {"Cs": 1}, "X": {"Br": 1}},
       "FAI":  {"A": {"FA": 1}, "X": {"I": 1}},   "MAI":  {"A": {"MA": 1}, "X": {"I": 1}},
       "MABr": {"A": {"MA": 1}, "X": {"Br": 1}},  "MACl": {"A": {"MA": 1}, "X": {"Cl": 1}},
       "PbI2": {"B": {"Pb": 1}, "X": {"I": 2}},   "PbBr2":{"B": {"Pb": 1}, "X": {"Br": 2}},
       "PbCl2":{"B": {"Pb": 1}, "X": {"Cl": 2}}}
MOLAR = {"m": 1.0, "mol": 1.0, "m total": 1.0, "mmol": 0.001}
MASS  = {"mg": 0.001, "g": 1.0, "g / 2ml": 1.0}
VOLATILE = {"MACl"}

def canon_precursor(s):
    k = re.sub(r"[^a-z0-9]", "", str(s).lower())
    for n in MW:
        if re.sub(r"[^a-z0-9]", "", n.lower()) == k:
            return n
    return None

def unit_family(u):
    u = str(u).strip().lower()
    return "molar" if u in MOLAR else ("mass" if u in MASS else None)

# J3: join ชั้น PSK กับสารตั้งต้น
psk = lay.loc[lay["layer_type"] == "PSK",
              ["device_id", "layer_id", "material"]].rename(columns={"material": "layer_material"})
jm = psk.merge(tp["materials"], on=["device_id", "layer_id"], how="left")
jm["prec"] = jm["material"].map(canon_precursor)
jm = jm.dropna(subset=["prec"]).copy()
jm = jm[~jm["prec"].isin(VOLATILE)]
jm["ufam"] = jm["unit"].map(unit_family)
jm["amt"]  = pd.to_numeric(jm["amount"], errors="coerce")

# J4: คำนวณจากสารตั้งต้นทีละ device
prec_comp, prec_status = {}, {}
for dev, g in jm.groupby("device_id"):
    fams = set(g["ufam"].dropna())
    bad  = g[g["ufam"].isna() | g["amt"].isna()]
    if len(bad):
        prec_status[dev] = "ตกหล่น: " + ", ".join(sorted(set(bad["prec"])))
    elif len(fams) != 1:
        prec_status[dev] = f"หน่วยปนตระกูล {sorted(fams)}"
    elif not any(ION[p].get("B") for p in g["prec"]):
        prec_status[dev] = "ไม่มี B-site (สูตรไม่ครบ)"
    else:
        fam, A, X = fams.pop(), {}, {}
        conv = MOLAR if fam == "molar" else MASS
        for _, r in g.iterrows():
            mol = r["amt"] * conv[str(r["unit"]).strip().lower()]
            if fam == "mass":
                mol /= MW[r["prec"]]
            spec = ION[r["prec"]]
            for i, mlt in spec.get("A", {}).items(): A[i] = A.get(i, 0) + mol * mlt
            for i, mlt in spec.get("X", {}).items(): X[i] = X.get(i, 0) + mol * mlt
        sA, sX = sum(A.values()), sum(X.values())
        prec_comp[dev] = ({k: v/sA for k, v in A.items()}, {k: v/sX for k, v in X.items()}, "Pb")
        prec_status[dev] = f"ok ({fam})"

# J5: รวม 2 แหล่ง สูตรเป็นหลัก precursor เติมช่องว่าง + cross-check
TOL = 0.03
rows = []
for _, r in psk.iterrows():
    dev = r["device_id"]
    f_res, p_res = composition_from_formula(r["layer_material"]), prec_comp.get(dev)
    row = {"device_id": dev, "layer_material": r["layer_material"],
           "precursor_status": prec_status.get(dev, "ไม่มีแถว precursor")}
    if f_res:
        A, X, b = f_res
        row["comp_source"] = "formula"
        if p_res:
            pA, pX, _ = p_res
            diff = [k for k in set(A) | set(X) | set(pA) | set(pX)
                    if abs({**A, **X}.get(k, 0) - {**pA, **pX}.get(k, 0)) > TOL]
            row["cross_check"] = "ตรง" if not diff else "ต่าง: " + ", ".join(sorted(diff))
    elif p_res:
        A, X, b = p_res
        row["comp_source"] = "precursor"
        row["cross_check"] = "-- (สูตรแตกไม่ได้)"
    else:
        row["comp_source"] = "ไม่มีข้อมูล"
        rows.append(row)
        continue
    row["b_site"] = b
    row.update({f"a_{k}": round(v, 3) for k, v in A.items()})
    row.update({f"x_{k}": round(v, 3) for k, v in X.items()})
    rows.append(row)

psk_composition = pd.DataFrame(rows)
front = ["device_id", "comp_source", "cross_check", "b_site"]
psk_composition = psk_composition[[c for c in front if c in psk_composition.columns]
                                  + [c for c in psk_composition.columns if c not in front]]

n_ok = int((psk_composition["comp_source"] != "ไม่มีข้อมูล").sum())
print(f"องค์ประกอบ {n_ok}/{psk_composition['device_id'].nunique()} devices | "
      f"source: {psk_composition['comp_source'].value_counts().to_dict()}")

# device-level สำหรับ merge เข้า target_table (ตามแผน §9: a_Cs / a_FA / x_Br)
# ไอออนที่ไม่อยู่ในสูตร = สัดส่วน 0 จริง ไม่ใช่ missing (เช่น pure iodide -> x_Br = 0)
# fill 0 เฉพาะ device ที่คำนวณองค์ประกอบได้ -- ที่คำนวณไม่ได้คง NaN
_cd = psk_composition.drop_duplicates("device_id").set_index("device_id")
comp_dev = _cd.reindex(columns=["a_Cs", "a_FA", "x_Br"])
_ok = _cd["comp_source"] != "ไม่มีข้อมูล"
comp_dev.loc[_ok] = comp_dev.loc[_ok].fillna(0.0)
print("coverage (หลัง fill 0 ให้ไอออนที่ไม่อยู่ในสูตร):", comp_dev.notna().sum().to_dict())

## 8.6 `material_family` ที่เป็น NaN — จัดการตามบทบาท ไม่ใช่เติมมั่ว

NaN ในแต่ละตาราง/ชั้นคนละความหมาย:

| ที่ | สถานะจริง | จัดการ |
|---|---|---|
| layers L4 (PSK) | NaN ทั้งหมด — material เป็นสูตรเคมี | มีตัวแทนที่ดีกว่าแล้ว: `psk_composition` (§8.5) |
| layers L1 / L2 / L8 | TCO / NiOx / Metal ล้วน — **variance = 0** | ใช้เป็น feature ไม่ได้ ปล่อยไว้ |
| layers L6 (ETL) | PCBM 24 / C60 18 / BCP 2 — **มี variance**, PCBM 9 paper / C60 5 paper | **ดึงเป็น `etl_family`** เข้า target_table (cell ล่าง) |
| materials | precursor / solvent / additive (CsI, FAI, DMF, ...) | บทบาทมีในคอลัมน์ `type` อยู่แล้ว — family ไม่ applicable |
| vendors | provenance | ปล่อย NaN ไม่ใช่ feature |

In [ ]:
# ---- §8.6 etl_family: ตัวเดียวใน material_family ที่มี variance พอเป็น feature ----
_etl = lay[(lay["layer_id"] == 6) & lay["material"].notna()]
assert (_etl.groupby("device_id").size() == 1).all(), "L6 มีหลายแถวต่อ device -- ต้องรวมก่อน"

arch_dev = _etl.set_index("device_id")[["material_family"]].rename(columns={"material_family": "etl_family"})
arch_dev["etl_family"] = arch_dev["etl_family"].fillna(_etl.set_index("device_id")["material"])

_ap = pd.Series(arch_dev.index, index=arch_dev.index).str.rsplit("_Sample", n=1).str[0]
print("etl_family:", arch_dev["etl_family"].value_counts(dropna=False).to_dict())
print("กระจาย paper:", {f: int(_ap[arch_dev["etl_family"] == f].nunique()) for f in arch_dev["etl_family"].unique()},
      "-- PCBM/C60 ข้ามหลาย paper ใช้ใต้ LOPO ได้ | BCP 2 dev/1 paper ระวังตอนตีความ")

## 8.7 ปิดช่องโหว่ `value_parsed` ไม่ตาม `value_corrected`

`column_prepare` §3.2 คำนวณ `value_parsed = to_numeric(value)` จาก**ค่าดิบ** ไม่เคยอ่าน `value_corrected`
ผล: การแก้ที่**เปลี่ยนตัวเลข** (ไม่ใช่แค่เปลี่ยนป้าย unit) หายไปเงียบ ๆ จากคอลัมน์ที่ทุกคนใช้

ตรวจ `unit_value_errors.csv` (27 แถว) ทีละกลุ่ม:

| กลุ่ม | จำนวน | สถานะ |
|---|---|---|
| Voc/Jsc ติดป้าย `%` | 12 | ✅ แก้แล้ว — เปลี่ยนแค่ป้าย unit ตัวเลขไม่เปลี่ยน `value_parsed` ถูกอยู่แล้ว |
| FF เป็นเศษส่วน | 8 | ✅ แก้แล้ว — ช่วง 3 คูณ 100 เข้า `value_parsed` โดยตรง |
| Hole mobility หน่วย m² | 3 | ⚠️ **แก้ครึ่งเดียว** — `value_corrected` = ×1e4 ถูก แต่ `value_parsed` ยังเป็นค่า m² เดิม |
| (false positive จาก regex เก่า) | 4 | หน่วยเป็น `cm2 V-1 s-1` อยู่แล้ว ไม่ต้องแก้ |

เซลล์ล่างซิงก์ `value_parsed` ให้ตาม `value_corrected` ทุกแถวที่ตัวเลขต่างกัน

In [ ]:
# ---- §8.7 ซิงก์ value_parsed ให้ตาม value_corrected (เขียนกลับเข้า tp["properties"]) ----
_pp = tp["properties"]
_v  = pd.to_numeric(_pp["value"], errors="coerce")
_vc = pd.to_numeric(_pp["value_corrected"], errors="coerce")
_vp = pd.to_numeric(_pp["value_parsed"], errors="coerce")

# แถวที่การแก้เปลี่ยน "ตัวเลข" จริง (ไม่ใช่แค่เปลี่ยนป้าย unit) แล้ว value_parsed ยังไม่ตาม
_need = (_vc.notna() & _v.notna() & ~np.isclose(_vc, _v, rtol=1e-9)
         & _vp.notna() & ~np.isclose(_vp, _vc, rtol=1e-9))
print(f"แถวที่ value_parsed ยังไม่ตาม value_corrected: {int(_need.sum())}")

# §7 (Y) รันไปก่อนหน้าแล้ว -- ต้องพิสูจน์ว่าไม่มีแถวไหนเป็น target/mediator ที่ใช้จริง
_used = set(TARGETS) | {"VTFL", "Trap density", "Rs", "Rrec"}
assert not _pp.loc[_need, "base_property"].isin(_used).any(),     "มีแถว target/mediator ถูกแก้ -- ต้องย้าย §8.7 ไปก่อน §7 แล้วรันใหม่"

if _need.any():
    display(_pp.loc[_need, ["device_id", "property", "value", "unit",
                            "unit_corrected", "value_corrected", "value_parsed"]])
    _pp.loc[_need, "value_parsed"] = _vc[_need]
    _pp.loc[_need, "value_note"] = (
        _pp.loc[_need, "value_note"].fillna("")
        + " | recheck §8.7: ซิงก์ value_parsed = value_corrected (m2 -> cm2 x1e4)").str.strip(" |")
    print("ซิงก์แล้ว -- ค่าใหม่:")
    display(_pp.loc[_need, ["device_id", "property", "unit_corrected", "value_parsed"]])

# gate: ไม่เหลือแถวที่ value_parsed ขัดกับ value_corrected
_chk = pd.to_numeric(_pp["value_parsed"], errors="coerce")
assert not (_vc.notna() & _v.notna() & ~np.isclose(_vc, _v, rtol=1e-9)
            & _chk.notna() & ~np.isclose(_chk, _vc, rtol=1e-9)).any(), "ยังมี value_parsed ไม่ตาม value_corrected"
print("gate ผ่าน: value_parsed สอดคล้องกับ value_corrected ทุกแถว")

### 8.8 แก้ `PSK optical band gap` จาก Figure S10b (SI)

ค่าที่กรอกไว้ของ Sample2/3/4 = 1.620 eV มาจาก **PL peak** ในตัวบท ("maximum PL ... now at 1.62 eV")
แต่คอลัมน์นี้คือ **optical band gap จาก Tauc plot** ซึ่ง SI Figure S10b ให้ค่ารายตัว:

| device | EMIMI | Tauc Eg (SI) | ที่กรอกไว้ |
|---|---|---|---|
| Sample1 | control | 1.609 | 1.609 ✅ |
| Sample2 | 1% | **1.605** | 1.620 ❌ |
| Sample3 | 2% | **1.615** | 1.620 ❌ |
| Sample4 | 4% | **1.616** | 1.620 ❌ |

In [ ]:
# ---- §8.8 แก้ PSK optical band gap ด้วยค่าจาก SI Figure S10b (Tauc plot) ----
PROP_FIX = {   # (device_id, property): ค่าที่ถูก
    ("10.1002admi.202500231_Sample2", "PSK optical band gap"): 1.605,
    ("10.1002admi.202500231_Sample3", "PSK optical band gap"): 1.615,
    ("10.1002admi.202500231_Sample4", "PSK optical band gap"): 1.616,
}
_pp = tp["properties"]
_n = 0
for (d, prop_name), good in PROP_FIX.items():
    m = (_pp["device_id"] == d) & (_pp["property"] == prop_name)
    assert m.sum() == 1, f"PROP_FIX จับได้ {int(m.sum())} แถว (ต้อง 1): {d} / {prop_name}"
    _pp.loc[m, "value_parsed"] = good
    _pp.loc[m, "value_note"] = (_pp.loc[m, "value_note"].fillna("")
        + " | recheck §8.8: ค่าเดิมเป็น PL peak แก้เป็น Tauc Eg จาก SI Figure S10b").str.strip(" |")
    _n += 1
print(f"แก้ optical band gap {_n} แถว (ยืนยันจาก SI Figure S10b)")
display(_pp[(_pp.device_id.str.startswith("10.1002admi")) & (_pp.property == "PSK optical band gap")]
        [["device_id", "value", "value_parsed", "value_note"]])

## 9. สรุปสิ่งที่ควรใช้

| ฝั่ง | ใช้ | จำนวน | หมายเหตุ |
|---|---|---|---|
| **Y ประสิทธิภาพ** | `voc` `jsc` `ff` แยก คูณเป็น pce | 43 dev / 15 paper | variant-consistent (`jv_variant` กำกับ champion/plain/average) |
| **mediator** | `vtfl` | ~28 dev / 11 paper | แทน grain_size (13) |
| **Y เสถียร** | `retention_25c` | 14 dev | exploratory เท่านั้น |
| **X knob** | `psk_spin_rpm` `psk_anneal_C` `psk_anneal_time_s` `antisolvent_vol_uL` **`antisolvent_drop_frac`** `htl_anneal_C` | 30-43 dev | §4-6.1 merge เข้า target_table แล้ว |
| **X ควบคุม** | `a_Cs` `a_FA` `x_Br` + `etl_family` | 44 dev | §8.5-8.6 กัน confound สูตรเคมี + สถาปัตยกรรม ETL |

**ตัดทิ้ง**: PCE ตรง (= ผลคูณ), grain_size แกนหลัก, t80, has_antisolvent/etl (paper เดียว), *_family variance=0 (ยกเว้น `etl_family` L6 — PCBM/C60 มี variance ข้าม paper เก็บใน §8.6)

**ระวัง**: ใช้ `antisolvent_drop_frac` (รวมสเกลแล้ว) ไม่ใช่ `antisolvent_timing_s` ดิบที่มี 2 จุดอ้างอิง — และอย่าใส่ทั้งคู่พร้อมกัน เป็นข้อมูลเดียวกัน

**โมเดล**: RandomForest + SHAP, LeaveOneGroupOut(paper) 15 fold, รายงาน mean±SD + bootstrap, ผล = hypothesis

In [ ]:
# ---- X: knob ระดับ device (PSK layer 4 + HTL layer 2) ----
# filter source_sheet: "NiOx synthesis" = สูตรทำผง NiOx ก่อนละลายเป็นน้ำหมึก ไม่ใช่ขั้นตอนสร้างเซลล์
# มีแค่ DOI เดียว (10.1002admi.202500231, 4 dev) -> confound 100% กับ paper นั้น ห้ามเป็น feature
_cell = pr_out[pr_out["source_sheet"] == "Process"]
_lidn = pd.to_numeric(_cell["layer_id"], errors="coerce")
_pskp = _cell[_lidn == 4]
_htlp = _cell[_lidn == 2]

def _knob(df, ptype, col, agg):
    if col not in df.columns:
        return pd.Series(dtype=float)
    g = df[df["process_type"] == ptype].groupby("device_id")[col]
    # min_count=1: กลุ่มที่เป็น NaN ล้วนต้องได้ NaN ไม่ใช่ 0
    # (pandas .sum() คืน 0.0 ให้กลุ่ม all-NaN -> อ่านเป็น "อบ 0 วินาที" ทั้งที่แปลว่า "ไม่รู้")
    return g.sum(min_count=1) if agg == "sum" else getattr(g, agg)()

X_dev = pd.DataFrame(index=device_layer_flags.set_index("device_id").index)
X_dev["psk_spin_rpm"]         = _knob(_pskp, "Spin coating", "Spin coating__RPM", "max")   # step สุดท้าย = RPM สูงสุด
X_dev["psk_anneal_C"]         = _knob(_pskp, "Annealing", "Annealing__C", "max")
X_dev["psk_anneal_time_s"]    = _knob(_pskp, "Annealing", "Annealing__time_s", "sum")
X_dev["antisolvent_vol_uL"]   = _knob(_pskp, "AntiSolvent", "AntiSolvent__vol_uL", "max")
X_dev["antisolvent_timing_s"] = _knob(_pskp, "AntiSolvent", "AntiSolvent__timing_s", "first")
X_dev["antisolvent_timing_ref"] = (_pskp.dropna(subset=["AntiSolvent__timing_s"])
                                   .groupby("device_id")["AntiSolvent__timing_ref"].first())
X_dev["htl_anneal_C"]         = _knob(_htlp, "Annealing", "Annealing__C", "max")

# ---- รวมสเกล antisolvent timing ให้เป็นแกนเดียว ----
# ดิบมี 2 จุดอ้างอิง: บวก = วินาทีหลังเริ่ม spin ขั้นสุดท้าย / ลบ = วินาทีก่อนจบ
# แปลงตัวลบด้วยความยาว spin ขั้นสุดท้าย -> ได้ "วินาทีนับจากเริ่ม spin ขั้นสุดท้าย" เหมือนกันหมด
_spin = (_pskp[(_pskp["process_type"] == "Spin coating") & _pskp["Spin coating__sec"].notna()]
         .sort_values("step_order").groupby("device_id").tail(1)
         .set_index("device_id")["Spin coating__sec"])
X_dev["psk_spin_last_s"] = _spin

_t = X_dev["antisolvent_timing_s"]
X_dev["antisolvent_drop_s"] = _t.where(_t >= 0, _spin.reindex(X_dev.index) + _t)
X_dev["antisolvent_drop_frac"] = X_dev["antisolvent_drop_s"] / X_dev["psk_spin_last_s"]

# gate: จุดหยดต้องอยู่ในช่วง spin จริง (0 < t <= ความยาว spin)
_chk = X_dev.dropna(subset=["antisolvent_drop_s", "psk_spin_last_s"])
assert ((_chk["antisolvent_drop_s"] > 0) & (_chk["antisolvent_drop_s"] <= _chk["psk_spin_last_s"])).all(),     "จุดหยด antisolvent หลุดนอกช่วง spin"

_n_conv = int((_t < 0).sum())
print(f"antisolvent_drop_s: {int(X_dev['antisolvent_drop_s'].notna().sum())} dev "
      f"(แปลงจาก before_end {_n_conv} dev) | frac: {int(X_dev['antisolvent_drop_frac'].notna().sum())} dev")
display(X_dev.loc[_t.notna(), ["psk_spin_last_s", "antisolvent_timing_s",
                               "antisolvent_timing_ref", "antisolvent_drop_s",
                               "antisolvent_drop_frac"]].round(3).head(12))

# target_table = flags + Y + stability + X knob + composition + สถาปัตยกรรม (etl_family §8.6)
target_table = (device_layer_flags.set_index("device_id")
                .join(Y).join(stab_Y).join(X_dev).join(comp_dev).join(arch_dev))

assert len(target_table) == 44 and target_table.index.is_unique, "target_table ต้อง 44 แถว unique"
print("target_table:", target_table.shape)
print("\ncoverage ต่อคอลัมน์ (devices):")
display(target_table.notna().sum().rename("n_devices").to_frame().T)
display(target_table.head())
# ยังไม่เขียน -- รวมเขียนทีเดียวใน §10

## 10. Export `prepared_v2` — ชุดเต็มทุกไฟล์

`prepared_v2/` = ทุกตารางจาก `prepared/` + สิ่งที่ recheck เพิ่ม

| ไฟล์ | เปลี่ยนจาก `prepared/` |
|---|---|
| `layers.csv` | + `layer_kind` (map ตรงจาก layer_id — **ไม่ merge** กันแถวพองเคส layer 7 สองแถว PCBM+BCP) |
| `process.csv` | + knob columns ที่แตก + `param_fixed` (§4.0) + `param_note` + `AntiSolvent__vol_uL/timing_s/timing_ref` (§6.1) + `*__time_s` (§6) |
| `devices.csv` | + layer flags (is_coated, coat_*, anneal_*) |
| `materials/properties/stability/vendors.csv` | เหมือนเดิม |
| `target_table.csv` | ใหม่ — Y (variant-consistent) + mediator + stability + **X knob + composition** |
| `device_layer_flags.csv` | ใหม่ |
| `psk_composition.csv` | ใหม่ — §8.5 (เดิมคำนวณใน column_prepare แต่ไม่เคย export) |

In [ ]:
V2 = ROOT / "processed" / "prepared_v2"
V2.mkdir(parents=True, exist_ok=True)

# 1) layers + layer_kind -- map ตรงจาก layer_id ห้าม merge
#    (solr_202400534 มี layer 7 สองแถว PCBM+BCP -- merge บน (device_id, layer_id) จะคูณแถวเป็น 2x2)
_lay_out = lay.copy()
_kind = _lay_out["layer_id"].map(
    lambda i: "main" if i in MAIN_IDS else "coating" if i in COATING_IDS else "other")
_lay_out["layer_kind"] = _kind.where(_lay_out["material"].notna())
# แถวที่เป็น additive (ผสมในน้ำหมึก ไม่ได้เคลือบทับ) ต้องไม่ถูกเรียกว่า coating -- ดู §2.1
for _lid, _col in MOD_KIND.items():        # ใช้ device_layer_flags -- `dev` ถูก loop ใน §7/§8.5 เขียนทับ
    _adv = device_layer_flags.loc[device_layer_flags[_col] == "additive", "device_id"]
    _m = (_lay_out["layer_id"] == _lid) & _lay_out["device_id"].isin(_adv)
    _lay_out.loc[_m, "layer_kind"] = "additive"
print("layer_kind:", _lay_out["layer_kind"].value_counts(dropna=False).to_dict())
assert len(_lay_out) == len(lay), "layers แถวพอง"
_lay_out.to_csv(V2 / "layers.csv", index=False)

# 2) process = แตก parameter + PARAM_FIX + antisolvent + time_s
assert int((pr_out["param_note"] != "").sum()) == 0, "ไม่ควรเหลือแถวติดธงแล้ว (ปิดด้วย SI)"
pr_out.to_csv(V2 / "process.csv", index=False)

# 3) devices + layer/anneal flags
_dev_out = tp["devices"].merge(device_layer_flags, on="device_id", how="left")
_dev_out.to_csv(V2 / "devices.csv", index=False)

# 4) ตารางที่เหลือ copy ตรง
for name in ["materials", "properties", "stability", "vendors"]:
    tp[name].to_csv(V2 / f"{name}.csv", index=False)

# 5) ตารางที่ recheck สร้าง
target_table.to_csv(V2 / "target_table.csv")
device_layer_flags.to_csv(V2 / "device_layer_flags.csv", index=False)
psk_composition.to_csv(V2 / "psk_composition.csv", index=False)

print("wrote prepared_v2:")
for p in sorted(V2.glob("*.csv")):
    d = pd.read_csv(p)
    print(f"  {p.name:24s} {d.shape}")

# verify จากไฟล์บนดิสก์
_chk_lay = pd.read_csv(V2 / "layers.csv")
assert len(_chk_lay) == len(lay), "layers บนดิสก์แถวไม่ตรง"
_chk_tt = pd.read_csv(V2 / "target_table.csv")
assert len(_chk_tt) == 44 and _chk_tt["device_id"].is_unique, "target_table บนดิสก์ไม่ใช่ 44 unique"
_hv = _chk_tt.dropna(subset=["voc", "jsc", "ff", "pce"])
_e = (_hv["voc"] * _hv["jsc"] * _hv["ff"] / 100 - _hv["pce"]).abs()
assert int((_e > 1.0).sum()) == 0, "ยังมี pce ปน variant"   # 1.0 เผื่อ rounding ค่ารายงาน (บั๊กเดิม err 1.64)
# gate: knob ใน target_table ต้องไม่มีค่าที่มาจาก NiOx synthesis (confound 100% กับ DOI เดียว)
_ns_dev = set(pr_out.loc[pr_out["source_sheet"] == "NiOx synthesis", "device_id"])
_KNOB = ["psk_spin_rpm", "psk_anneal_C", "psk_anneal_time_s",
         "antisolvent_vol_uL", "antisolvent_timing_s", "htl_anneal_C"]
_ref = (_chk_tt.set_index("device_id")[_KNOB]
        .compare(target_table[_KNOB]) if False else None)   # (เทียบตรงผ่าน assert ด้านล่างแทน)
_cell_only = pr_out[pr_out["source_sheet"] == "Process"]
_lidn2 = pd.to_numeric(_cell_only["layer_id"], errors="coerce")
_htl_ref = (_cell_only[(_lidn2 == 2) & (_cell_only["process_type"] == "Annealing")]
            .groupby("device_id")["Annealing__C"].max())
_htl_now = _chk_tt.set_index("device_id")["htl_anneal_C"].dropna()
assert _htl_now.round(6).equals(_htl_ref.reindex(_htl_now.index).round(6)),     "htl_anneal_C ไม่ตรงกับค่าที่คำนวณจาก source_sheet='Process' อย่างเดียว"
print(f"gate NiOx synthesis ผ่าน: knob ทั้ง {len(_KNOB)} ตัวมาจาก source_sheet='Process' เท่านั้น "
      f"({len(_ns_dev)} device ที่มีชีตสังเคราะห์ไม่ปนเข้ามา)")

print(f"verify ผ่าน: layers {len(_chk_lay)} แถวไม่พอง | target_table 44 unique | pce consistency max_err={_e.max():.3f}")